# Figure 1 — Subject-specific language fROI masks

4-panel glass brain layout per subject:
- **a)** Averaged across sessions  
- **b)** Session 1  
- **c)** Session 2  
- **d)** Session 3

Each panel shows 4 task maps (aliceFr, aliceEn, listening, reading).

**Inputs** (from env vars injected by `invoke run-notebooks`):
- `OUTPUT_DATA_DIR` — pipeline output root
- `SOURCE_DATA_DIR` — pipeline source root (for atlas)


In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from nilearn import plotting

# ---------------------------------------------------------------------------
# Paths — injected by airoh run_notebooks as environment variables
# ---------------------------------------------------------------------------
OUTPUT_DIR = Path(os.environ.get('OUTPUT_DATA_DIR', 'output_data'))
SOURCE_DIR = Path(os.environ.get('SOURCE_DATA_DIR', 'source_data'))

LANGLOC_DIR = OUTPUT_DIR / 'langlocalizer'
FROI_DIR    = LANGLOC_DIR / 'fedorenko_frois'

FIGURES_OUTPUT_DIR = Path(os.environ.get('FIGURES_OUTPUT_DIR',
                          str(OUTPUT_DIR / 'figures_manuscript')))
FIG_DIR = FIGURES_OUTPUT_DIR / 'fig1_froi_masks'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'Output dir : {OUTPUT_DIR}')
print(f'fROI dir   : {FROI_DIR}')
print(f'Figure dir : {FIG_DIR}')

Output dir : /scratch/ibilgin/cneuromod.glm/output_data
fROI dir   : /scratch/ibilgin/cneuromod.glm/output_data/langlocalizer/fedorenko_frois
Figure dir : /scratch/ibilgin/cneuromod.glm/output_data/figures_manuscript/fig1_froi_masks


In [2]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
TOP_PERCENT  = 0.10   # must match froi_top_percent in invoke.yaml
DISPLAY_MODE = 'l'    # left-hemisphere sagittal view

TASKS = ['aliceFr', 'aliceEn', 'listening', 'reading']
TASK_CONTRASTS = {
    'aliceFr':   'int-degr',
    'aliceEn':   'int-degr',
    'listening': 'int-degr',
    'reading':   'word-nonword',
}
TASK_LABELS = {
    'aliceFr':   'aliceFR',
    'aliceEn':   'aliceEN',
    'listening': 'listening',
    'reading':   'reading',
}
TASK_COLORS = {
    'aliceFr':   '#2ca02c',
    'aliceEn':   '#1f77b4',
    'listening': '#9467bd',
    'reading':   '#d62728',
}
TASK_CMAPS = {
    'aliceFr':   'Greens',
    'aliceEn':   'Blues',
    'listening': 'Purples',
    'reading':   'Reds',
}
PANEL_LABELS = {'across': 'a)', 'ses-001': 'b)', 'ses-002': 'c)', 'ses-003': 'd)'}
PANEL_TITLES = {
    'across':  'Averaged across sessions',
    'ses-001': 'Session 1',
    'ses-002': 'Session 2',
    'ses-003': 'Session 3',
}

In [3]:
# ---------------------------------------------------------------------------
# Data helpers
# ---------------------------------------------------------------------------

def discover_subjects():
    """Return sorted list of subjects that have fROI outputs."""
    subjects = sorted([
        p.name for p in FROI_DIR.glob('sub-*')
        if p.is_dir()
    ])
    # Also check flat fROI files (subject-level files are in FROI_DIR directly)
    if not subjects:
        subjects = sorted(set(
            f.name.split('_')[0]
            for f in FROI_DIR.glob('sub-*_task-*_parcel-*_top_mean.nii.gz')
        ))
    return subjects


def find_subject_froi_mask(subject, session_key, task):
    """
    Find the combined (all-parcel union) fROI mask for a subject/session/task.

    session_key : 'across' | 'ses-001' | 'ses-002' | ...

    Returns a Nifti1Image (union of all parcel top-voxel masks) or None.
    """
    contrast = TASK_CONTRASTS[task]

    if session_key == 'across':
        # Subject-level fROIs: _top_mean.nii.gz
        pattern = f'{subject}_task-{task}_contrast-{contrast}_parcel-*_top_mean.nii.gz'
        files = sorted(FROI_DIR.glob(pattern))
    else:
        # Session-level fROIs: _top.nii.gz
        pattern = f'{subject}_{session_key}_task-{task}_contrast-{contrast}_parcel-*_top.nii.gz'
        files = sorted(FROI_DIR.glob(pattern))

    if not files:
        return None

    ref   = nib.load(str(files[0]))
    union = np.zeros(ref.shape, dtype=np.uint8)
    for f in files:
        img = nib.load(str(f))
        union |= (img.get_fdata() > 0).astype(np.uint8)

    return nib.Nifti1Image(union, ref.affine, ref.header)


def render_brain(ax, img, cmap):
    """Plot a glass brain onto ax; show placeholder if img is None."""
    if img is None:
        ax.text(0.5, 0.5, 'n/a', ha='center', va='center',
                fontsize=8, color='#aaaaaa', transform=ax.transAxes)
        ax.set_facecolor('white')
        ax.axis('off')
        return
    plotting.plot_glass_brain(
        img, axes=ax,
        display_mode=DISPLAY_MODE,
        cmap=cmap,
        bg_img=None,
        black_bg=False,
        plot_abs=False,
        colorbar=False,
        threshold=0.1,
        alpha=0.85,
        vmax=1,
        annotate=False,
    )
    ax.set_facecolor('white')

In [4]:
# ---------------------------------------------------------------------------
# Figure builder
# ---------------------------------------------------------------------------

def make_figure1(subject, dpi=200):
    """
    Build Figure 1 for one subject: 2x2 panels, each with 4 task brain maps.
    Saves to FIG_DIR and returns the output path.
    """
    panels = [
        ('across',  0, 0),
        ('ses-001', 0, 1),
        ('ses-002', 1, 0),
        ('ses-003', 1, 1),
    ]

    fig = plt.figure(figsize=(10, 9), facecolor='white')
    outer = gridspec.GridSpec(
        2, 2, figure=fig,
        hspace=0.32, wspace=0.10,
        left=0.02, right=0.98,
        top=0.90, bottom=0.02,
    )

    for session_key, row, col in panels:
        # Panel border
        panel_ax = fig.add_subplot(outer[row, col])
        panel_ax.set_facecolor('white')
        panel_ax.set_xticks([])
        panel_ax.set_yticks([])
        for sp in panel_ax.spines.values():
            sp.set_linewidth(0.8)
            sp.set_edgecolor('#999999')
        panel_ax.set_title(
            f"{PANEL_LABELS[session_key]}   {PANEL_TITLES[session_key]}",
            fontsize=10, loc='left', pad=6, color='black',
        )

        # Inner 4-row x 2-col grid: alternating label/brain rows
        inner = gridspec.GridSpecFromSubplotSpec(
            4, 2,
            subplot_spec=outer[row, col],
            height_ratios=[0.12, 1, 0.12, 1],
            hspace=0.06, wspace=0.04,
        )

        task_positions = [
            ('aliceFr',   0, 1, 0),
            ('aliceEn',   0, 1, 1),
            ('listening', 2, 3, 0),
            ('reading',   2, 3, 1),
        ]

        for task, label_row, brain_row, col_idx in task_positions:
            # Task label
            lax = fig.add_subplot(inner[label_row, col_idx])
            lax.set_facecolor('white')
            lax.axis('off')
            lax.text(
                0.5, 0.05, TASK_LABELS[task],
                ha='center', va='bottom', fontsize=10,
                color=TASK_COLORS[task], fontweight='bold',
                transform=lax.transAxes,
            )

            # Brain
            ax = fig.add_subplot(inner[brain_row, col_idx])
            ax.set_facecolor('white')
            ax.set_xticks([])
            ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_visible(False)

            img = find_subject_froi_mask(subject, session_key, task)
            render_brain(ax, img, TASK_CMAPS[task])

    fig.suptitle(
        f"{subject}  —  Subject-specific language fROIs  "
        f"(top {int(TOP_PERCENT*100)}% per parcel)",
        fontsize=11, fontweight='bold', y=0.975, color='black',
    )

    out_path = FIG_DIR / f"{subject}_fig1_froi_masks.png"
    fig.savefig(str(out_path), dpi=dpi, bbox_inches='tight',
                facecolor='white', pad_inches=0.15)
    plt.close(fig)
    print(f'  Saved: {out_path.name}')
    return out_path

In [5]:
# ---------------------------------------------------------------------------
# Run
# ---------------------------------------------------------------------------
subjects = discover_subjects()
print(f'Subjects with fROI outputs: {subjects}')

if not subjects:
    print('[SKIP] No fROI outputs found — run `invoke run-froi` first.')
else:
    for subject in subjects:
        print(f'\n{subject}:')
        make_figure1(subject)

    print(f'\nFigures saved to: {FIG_DIR}')

Subjects with fROI outputs: ['sub-01', 'sub-02', 'sub-03', 'sub-05']

sub-01:


  Saved: sub-01_fig1_froi_masks.png

sub-02:


  Saved: sub-02_fig1_froi_masks.png

sub-03:


  Saved: sub-03_fig1_froi_masks.png

sub-05:


  Saved: sub-05_fig1_froi_masks.png

Figures saved to: /scratch/ibilgin/cneuromod.glm/output_data/figures_manuscript/fig1_froi_masks
